In [1]:
import pandas as pd
import glob
import os
import torch
import numpy as np
from datetime import datetime
from scipy.stats import pearsonr

# 获取所有 CSV 文件
data_dir = 'NASDAQ/'
csv_files = glob.glob(os.path.join(data_dir, 'NASDAQ_*_30Y.csv'))

# 加载公司数据
companies_data = {}
for f in csv_files:
    try:
        company_name = os.path.basename(f).split('_')[1]
        df = pd.read_csv(f)
        df['Date'] = pd.to_datetime(df['Date'], utc=True)
        df['Date'] = df['Date'].dt.tz_convert(None)
        companies_data[company_name] = df
    except Exception as e:
        print(f"Error loading {f}: {e}")

# 定义日期范围
train_start = datetime.strptime('2013-01-01', '%Y-%m-%d')
train_end = datetime.strptime('2013-8-31', '%Y-%m-%d')
val_start = datetime.strptime('2013-9-01', '%Y-%m-%d')
val_end = datetime.strptime('2013-10-31', '%Y-%m-%d')
test_start = datetime.strptime('2013-11-01', '%Y-%m-%d')
test_end = datetime.strptime('2013-12-31', '%Y-%m-%d')

# 按日期范围划分数据
train_data = {}
val_data = {}
test_data = {}
for company, df in companies_data.items():
    train_data[company] = df[(df['Date'] >= train_start) & (df['Date'] <= train_end)][['Date', 'Open', 'High', 'Low', 'Close', 'Volume']].copy()
    val_data[company] = df[(df['Date'] >= val_start) & (df['Date'] <= val_end)][['Date', 'Open', 'High', 'Low', 'Close', 'Volume']].copy()
    test_data[company] = df[(df['Date'] >= test_start) & (df['Date'] <= test_end)][['Date', 'Open', 'High', 'Low', 'Close', 'Volume']].copy()

company_names = list(companies_data.keys())
num_companies = len(company_names)

In [ ]:
def pearsonr_torch(x, y, device='cuda'):
    """
    计算两个序列的皮尔逊相关系数（GPU 加速）
    x, y: shape (seq_len,) 的 PyTorch 张量
    """
    x = x.to(device)
    y = y.to(device)
    mean_x = torch.mean(x)
    mean_y = torch.mean(y)
    xm = x - mean_x
    ym = y - mean_y
    cov_xy = torch.sum(xm * ym)
    std_x = torch.sqrt(torch.sum(xm ** 2))
    std_y = torch.sqrt(torch.sum(ym ** 2))
    corr = cov_xy / (std_x * std_y + 1e-10)  # 避免除以零
    return corr

def create_graph_data(dataset, company_names, seq_len, save_dir, split, corr_threshold=0.7, device='cuda'):
    if not os.path.exists(os.path.join(save_dir, split)):
        os.makedirs(os.path.join(save_dir, split))
    
    num_companies = len(company_names)
    num_windows = 0
    
    # 检查是否有可用的 GPU
    if device == 'cuda' and not torch.cuda.is_available():
        print("警告：未检测到 GPU，使用 CPU 进行计算")
        device = 'cpu'
    
    # 确定最小数据长度
    min_length = min(len(dataset[company]) for company in company_names if not dataset[company].empty)
    if min_length < seq_len + 1:
        print(f"警告：{split} 集数据不足 (min_length={min_length})")
        return 0
    
    # 按行索引生成时间窗口
    for i in range(min_length - seq_len):
        X = []
        Y = []
        valid_companies = []
        close_prices_window = []
        
        # 收集当前窗口的数据
        for j, company in enumerate(company_names):
            df = dataset[company]
            if len(df) < seq_len + 1:
                continue
            
            window_data = df.iloc[i:i+seq_len]
            if len(window_data) == seq_len:
                features = window_data[['Open', 'High', 'Low', 'Close', 'Volume']].values
                X.append(features)
                close_prices_window.append(window_data['Close'].values)
                
                current_close = window_data['Close'].iloc[-1]
                next_data = df.iloc[i+seq_len] if i+seq_len < len(df) else None
                if next_data is not None:
                    next_close = next_data['Close']
                    label = 1 if next_close > current_close else 0
                    Y.append(label)
                    valid_companies.append(j)
        
        # 仅当所有公司数据完整时继续
        if len(X) == num_companies and len(Y) == num_companies:
            # 计算当前窗口的相关性矩阵
            adj_matrix = np.zeros((num_companies, num_companies))
            close_prices_window = np.array(close_prices_window)  # (num_companies, seq_len)
            close_prices_tensor = torch.tensor(close_prices_window, dtype=torch.float32)
            
            for m in range(num_companies):
                for n in range(m + 1, num_companies):
                    if len(close_prices_window[m]) > 1 and len(close_prices_window[n]) > 1:
                        corr = pearsonr_torch(close_prices_tensor[m], close_prices_tensor[n], device)
                        if abs(corr) > corr_threshold:
                            adj_matrix[m, n] = 1
                            adj_matrix[n, m] = 1
            
            # 生成 edge_index
            indices = np.vstack(np.where(adj_matrix > 0))
            edge_index = torch.from_numpy(indices).long()
            
            # 保存图数据
            X = torch.from_numpy(np.array(X, dtype=np.float32))  # (num_companies, seq_len, 5)
            Y = torch.tensor(Y, dtype=torch.long)   # (num_companies,)
            graph_data = {'X': X, 'A': edge_index, 'Y': Y}
            torch.save(graph_data, os.path.join(save_dir, split, f'graph_{num_windows}.pt'))
            num_windows += 1
    
    return num_windows

# 创建并保存图数据
save_dir = './w_5_NASDAQ_graph_data'
seq_len = 5
corr_threshold = 0.7
device = 'cuda' if torch.cuda.is_available() else 'cpu'
train_windows = create_graph_data(train_data, company_names, seq_len, save_dir=save_dir, split='train', corr_threshold=corr_threshold, device=device)
# val_windows = create_graph_data(val_data, company_names, seq_len, save_dir=save_dir, split='val', corr_threshold=corr_threshold, device=device)
# test_windows = create_graph_data(test_data, company_names, seq_len, save_dir=save_dir, split='test', corr_threshold=corr_threshold, device=device)
# print(f"创建了 {train_windows} 个训练图、{val_windows} 个验证图、{test_windows} 个测试图数据文件。")

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv
import os
import time
from sklearn.metrics import f1_score, matthews_corrcoef
import numpy as np

# 检查 GPU 可用性
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

class GATGRUModel(nn.Module):
    def __init__(self, input_dim, gru_hidden_dim, gat_hidden_dim, num_heads, output_dim=2):
        super(GATGRUModel, self).__init__()
        self.gru = nn.GRU(input_dim, gru_hidden_dim, batch_first=True)
        self.gat1 = GATConv(gru_hidden_dim, gat_hidden_dim, heads=num_heads, concat=True)
        self.gat2 = GATConv(gat_hidden_dim * num_heads, gat_hidden_dim, heads=1, concat=True)
        self.fc = nn.Linear(gat_hidden_dim, output_dim)

    def forward(self, x, edge_index):
        gru_out, _ = self.gru(x)
        gru_out = gru_out[:, -1, :]
        gat_out = F.relu(self.gat1(gru_out, edge_index))
        gat_out = self.gat2(gat_out, edge_index)
        out = self.fc(gat_out)
        return out

def train_model(model, save_dir, train_windows, val_windows, test_windows, epochs=600):
    # 将模型移动到 GPU
    model = model.to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    start_time = time.time()  # 记录训练开始时间
    best_val_accuracy = 0.0  # Track the best validation accuracy
    
    for epoch in range(epochs):
        epoch_start = time.time()  # 记录该 epoch 开始时间

        # 训练
        model.train()
        train_loss, train_accuracy, train_count = 0.0, 0.0, 0
        train_predictions, train_labels = [], []
        
        for i in range(train_windows):
            graph_data = torch.load(os.path.join(save_dir, 'train', f'graph_{i}.pt'))
            X, edge_index, Y = graph_data['X'], graph_data['A'], graph_data['Y']
            X, edge_index, Y = X.to(device), edge_index.to(device), Y.to(device)
            
            optimizer.zero_grad()
            out = model(X, edge_index)
            loss = criterion(out, Y)
            loss.backward()
            optimizer.step()
            
            _, predicted = torch.max(out, 1)
            train_loss += loss.item()
            train_accuracy += (predicted == Y).float().mean().item()
            train_count += 1
            
            train_predictions.extend(predicted.cpu().numpy())
            train_labels.extend(Y.cpu().numpy())
            
        # 验证
        model.eval()
        val_loss, val_accuracy, val_count = 0.0, 0.0, 0
        # --- New lists for validation predictions and labels ---
        val_predictions, val_labels = [], []
        
        with torch.no_grad():
            for i in range(val_windows):
                graph_data = torch.load(os.path.join(save_dir, 'val', f'graph_{i}.pt'))
                X, edge_index, Y = graph_data['X'], graph_data['A'], graph_data['Y']
                X, edge_index, Y = X.to(device), edge_index.to(device), Y.to(device)
                
                out = model(X, edge_index)
                loss = criterion(out, Y)
                _, predicted = torch.max(out, 1)
                val_loss += loss.item()
                val_accuracy += (predicted == Y).float().mean().item()
                val_count += 1
                
                val_predictions.extend(predicted.cpu().numpy())
                val_labels.extend(Y.cpu().numpy())
        
        # 平均指标
        avg_train_loss = train_loss / train_count if train_count > 0 else 0
        avg_train_accuracy = train_accuracy / train_count if train_count > 0 else 0
        avg_val_loss = val_loss / val_count if val_count > 0 else 0
        avg_val_accuracy = val_accuracy / val_count if val_count > 0 else 0
        
        train_f1 = f1_score(train_labels, train_predictions, average='weighted')
        train_mcc = matthews_corrcoef(train_labels, train_predictions)
        val_f1 = f1_score(val_labels, val_predictions, average='weighted')
        val_mcc = matthews_corrcoef(val_labels, val_predictions)
        
        if avg_val_accuracy > best_val_accuracy:
            torch.save(model.state_dict(), 'GAT+GRU_w5_gru64_gat32_head3.pt')
            best_val_accuracy = avg_val_accuracy  
            
        # 时间统计
        epoch_time = time.time() - epoch_start
        total_time = time.time() - start_time
        
        print(f'Epoch {epoch+1}, '
              f'Train Loss: {avg_train_loss:.4f}, Train Acc: {avg_train_accuracy:.4f}, '
              f'Train F1: {train_f1:.4f}, Train MCC: {train_mcc:.4f}, '
              f'Val Loss: {avg_val_loss:.4f}, Val Acc: {avg_val_accuracy:.4f}, '
              f'Val F1: {val_f1:.4f}, Val MCC: {val_mcc:.4f}, '
              f'Epoch Time: {epoch_time:.2f}s, Total Time: {total_time:.2f}s')
    
    print(f'Best Val Accuracy: {best_val_accuracy:.4f}')

# 参数
input_dim = 5  # Open, High, Low, Close, Volume
gru_hidden_dim = 64
gat_hidden_dim = 32
num_heads = 3

save_dir = './w_5_NYSE_graph_data'
train_windows = 139
val_windows = 11
test_windows = 35

# 初始化模型
model = GATGRUModel(input_dim, gru_hidden_dim, gat_hidden_dim, num_heads, output_dim=2)

train_model(model, save_dir, train_windows, val_windows, test_windows)

Using device: cuda
Epoch 1, Train Loss: 0.7053, Train Acc: 0.5195, Train F1: 0.5179, Train MCC: 0.0339, Val Loss: 0.6960, Val Acc: 0.5422, Val F1: 0.3814, Val MCC: 0.0009, Epoch Time: 24.79s, Total Time: 24.79s
Epoch 2, Train Loss: 0.6995, Train Acc: 0.5015, Train F1: 0.4651, Train MCC: -0.0235, Val Loss: 0.6912, Val Acc: 0.5422, Val F1: 0.3814, Val MCC: 0.0009, Epoch Time: 3.11s, Total Time: 27.90s
Epoch 3, Train Loss: 0.6971, Train Acc: 0.5133, Train F1: 0.4653, Train MCC: 0.0002, Val Loss: 0.6898, Val Acc: 0.5422, Val F1: 0.3814, Val MCC: 0.0009, Epoch Time: 3.08s, Total Time: 30.98s
Epoch 4, Train Loss: 0.6961, Train Acc: 0.5133, Train F1: 0.4653, Train MCC: 0.0002, Val Loss: 0.6906, Val Acc: 0.5422, Val F1: 0.3814, Val MCC: 0.0009, Epoch Time: 3.06s, Total Time: 34.04s
Epoch 5, Train Loss: 0.6943, Train Acc: 0.5210, Train F1: 0.3974, Train MCC: 0.0053, Val Loss: 0.6902, Val Acc: 0.5422, Val F1: 0.3814, Val MCC: 0.0009, Epoch Time: 3.05s, Total Time: 37.09s
Epoch 6, Train Loss: 0.6

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv
import os
import time
from sklearn.metrics import f1_score, matthews_corrcoef
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

class GATGRUModel(nn.Module):
    def __init__(self, input_dim, gru_hidden_dim, gat_hidden_dim, num_heads, output_dim=2):
        super(GATGRUModel, self).__init__()
        self.gru = nn.GRU(input_dim, gru_hidden_dim, batch_first=True)
        self.gat1 = GATConv(gru_hidden_dim, gat_hidden_dim, heads=num_heads, concat=True)
        self.gat2 = GATConv(gat_hidden_dim * num_heads, gat_hidden_dim, heads=1, concat=True)
        self.fc = nn.Linear(gat_hidden_dim, output_dim)

    def forward(self, x, edge_index):
        gru_out, _ = self.gru(x)
        gru_out = gru_out[:, -1, :]
        gat_out = F.relu(self.gat1(gru_out, edge_index))
        gat_out = self.gat2(gat_out, edge_index)
        out = self.fc(gat_out)
        return out

input_dim = 5  # Open, High, Low, Close, Volume
gru_hidden_dim = 64
gat_hidden_dim = 32
num_heads = 3

save_dir = './w_5_NYSE_graph_data'
train_windows = 139
val_windows = 11
test_windows = 35

model = GATGRUModel(input_dim, gru_hidden_dim, gat_hidden_dim, num_heads, output_dim=2)
model.load_state_dict(torch.load('GAT+GRU_w5_gru64_gat32_head3.pt', map_location=device))
model.to(device)
model.eval()  
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

val_loss, val_accuracy, val_count = 0.0, 0.0, 0
val_predictions, val_labels = [], []
with torch.no_grad():
    for i in range(val_windows):
        graph_data = torch.load(os.path.join(save_dir, 'val', f'graph_{i}.pt'))
        X, edge_index, Y = graph_data['X'], graph_data['A'], graph_data['Y']
        X, edge_index, Y = X.to(device), edge_index.to(device), Y.to(device)
        
        out = model(X, edge_index)
        loss = criterion(out, Y)
        _, predicted = torch.max(out, 1)
        val_loss += loss.item()
        val_accuracy += (predicted == Y).float().mean().item()
        val_count += 1
        
        val_predictions.extend(predicted.cpu().numpy())
        val_labels.extend(Y.cpu().numpy())

# 平均指标
avg_val_loss = val_loss / val_count if val_count > 0 else 0
avg_val_accuracy = val_accuracy / val_count if val_count > 0 else 0

# --- Calculate F1-score and MCC ---
val_f1 = f1_score(val_labels, val_predictions, average='weighted')
val_mcc = matthews_corrcoef(val_labels, val_predictions)

# --- Updated print statement with F1 and MCC ---
print(
      f'Val Loss: {avg_val_loss:.4f}, Val Acc: {avg_val_accuracy:.4f}, '
      f'Val F1: {val_f1:.4f}, Val MCC: {val_mcc:.4f} ')

Using device: cuda
Val Loss: 0.6753, Val Acc: 0.6202, Val F1: 0.6034, Val MCC: 0.2266 
